# 🏥 RSNA Knee Abnormality Detection — Exploratory Data Analysis (EDA)
---
Bu notebook, RSNA Knee yarışmasının tüm veri katmanlarını derinlemesine incelemek üzere tasarlanmıştır:
1. **Genel Veri Seti Yapısı:** `train.csv`, `train_series.csv`, `test.csv`, `test_series.csv`
2. **Etiket Mimarisi:** 58 Altın (Gold) Etiket vs 4.349 Etiketsiz Rapor
3. **Tıbbi Rapor Analizi:** İspanyolca radyoloji metinleri ve kritik semantik kalıplar
4. **MR Düzlemleri ve Serileri:** Sagittal, Coronal, Axial dağılımları ve sekans parametreleri
5. **2.5D Pencereleme (Windowing):** 3 ardışık dilimi RGB kanalına eşleme mantığı

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Grafik ayarları
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
print('Veri klasörü:', DATA_DIR.resolve())
print('Mevcut dosyalar:', [f.name for f in DATA_DIR.glob('*')])

## 1. Veri Setlerini Yükleme ve Boyut Kontrolü

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'train_df shape     : {train_df.shape}')
print(f'train_series shape : {train_series.shape}')
print(f'test_df shape      : {test_df.shape}')
print(f'test_series shape  : {test_series.shape}')
print(f'sample_sub shape   : {sample_sub.shape}')

TARGET_COLS = [c for c in sample_sub.columns if c != 'StudyInstanceUID']
print(f'\nToplam {len(TARGET_COLS)} Hedef Sınıf:')
for i, t in enumerate(TARGET_COLS, 1):
    print(f'  {i:2d}. {t}')

## 2. "58 Altın Etiket" vs "4.349 Rapor" Analizi
Veri setindeki en önemli kritik kırılım: Sadece 58 vakada el ile doğrulanmış altın etiketler vardır.

In [ ]:
gold_mask = train_df['ACL'].notna()
gold_df = train_df[gold_mask].copy()
unlabeled_df = train_df[~gold_mask].copy()

print(f'Altın (Gold-Standard) Etiketli Vaka : {len(gold_df)} (%{len(gold_df)/len(train_df)*100:.2f})')
print(f'Sadece Raporu Olan Vaka             : {len(unlabeled_df)} (%{len(unlabeled_df)/len(train_df)*100:.2f})')

# Hedef sınıfların 58 altın vaka içindeki prevalansı
target_means = gold_df[TARGET_COLS].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=target_means.values * 100, y=target_means.index, palette='viridis')
plt.title('58 Altın Vakadaki Pozitiflik Oranları (%)', fontsize=14, weight='bold')
plt.xlabel('Pozitif Oranı (%)')
for i, v in enumerate(target_means.values * 100):
    plt.text(v + 1, i, f'{v:.1f}% ({int(round(v*len(gold_df)/100))} vaka)', va='center')
plt.xlim(0, 80)
plt.tight_layout()
plt.show()

### Etiket Eş-Oluşum Matrisi (Co-occurrence Matrix)
Diz patolojileri birbirini tetikler (örn. ACL yırtığına sıklıkla menisküs yırtığı ve efüzyon eşlik eder).

In [ ]:
plt.figure(figsize=(10, 8))
corr = gold_df[TARGET_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, square=True)
plt.title('Hedef Hastalıklar Arası Korelasyon Matrisi (58 Altın Vaka)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

## 3. Radyoloji Raporları (Spanish NLP) Analizi
4.349 etiketsiz vakanın hepsinde İspanyolca doktor raporu bulunmaktadır.

In [ ]:
train_df['char_count'] = train_df['Report'].astype(str).str.len()
train_df['word_count'] = train_df['Report'].astype(str).apply(lambda x: len(x.split()))

print(train_df[['char_count', 'word_count']].describe())

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_df['word_count'], bins=40, kde=True, ax=ax[0], color='teal')
ax[0].set_title('Rapor Kelime Sayısı Dağılımı')
ax[0].set_xlabel('Kelime Sayısı')

sns.histplot(train_df['char_count'], bins=40, kde=True, ax=ax[1], color='coral')
ax[1].set_title('Rapor Karakter Sayısı Dağılımı')
ax[1].set_xlabel('Karakter Sayısı')
plt.tight_layout()
plt.show()

print('\n--- ÖRNEK RAPOR 1 ---')
print(train_df['Report'].iloc[0])
print('\n--- ÖRNEK RAPOR 2 ---')
print(train_df['Report'].iloc[1])

## 4. MR Serileri ve Anatomik Düzlem (Plane) Dağılımı
Her hasta için farklı düzlemlerde birden fazla MR serisi vardır.

In [ ]:
series_per_study = train_series.groupby('StudyInstanceUID')['SeriesInstanceUID'].count()
print(f'Hasta başına ortalama seri sayısı : {series_per_study.mean():.2f}')
print(f'Minimum seri sayısı              : {series_per_study.min()}')
print(f'Maksimum seri sayısı             : {series_per_study.max()}')

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
train_series['Anatomical_Plane'].value_counts().plot(kind='bar', ax=ax[0], color='cornflowerblue')
ax[0].set_title('Anatomik Düzlem Dağılımı')
ax[0].set_ylabel('Seri Sayısı')

train_series['Fluid_Sensitive'].value_counts().plot(kind='bar', ax=ax[1], color='lightcoral')
ax[1].set_title('Fluid Sensitive Dağılımı')

train_series['Fat_Suppression'].value_counts().plot(kind='bar', ax=ax[2], color='mediumseagreen')
ax[2].set_title('Fat Suppression Dağılımı')
plt.tight_layout()
plt.show()

## 5. 2.5D Pencereleme (3-Slice RGB Windowing) Stratejisi
Deep Search bulgularında gördüğümüz gibi:
* 3 ardışık dilim alınıp bir pencere (RGB) oluşturulur.
* Bu sayede 2D ImageNet modelleri (CoAtNet, ResNet) z-ekseni sürekliliğini yakalar.

In [ ]:
def demonstrate_25d_windowing(num_slices=30):
    """Bir serideki dilimlerin nasıl 3'lü pencerelere bölündüğünü simüle eder."""
    windows = []
    # İlk ve son %10 gürültü dilimlerini atlama simülasyonu
    start = int(num_slices * 0.1)
    end = int(num_slices * 0.9)
    valid_slices = list(range(start, end))
    
    for i in range(1, len(valid_slices) - 1):
        # 3 ardışık dilim: [slice_{i-1}, slice_i, slice_{i+1}]
        window = (valid_slices[i-1], valid_slices[i], valid_slices[i+1])
        windows.append(window)
        
    print(f'Toplam Dilim: {num_slices} -> Geçerli Dilim Aralığı: [{start}, {end}] ({len(valid_slices)} dilim)')
    print(f'Üretilen 2.5D RGB Pencere Sayısı: {len(windows)}')
    print(f'İlk 3 Pencere (RGB Kanalları): {windows[:3]}')
    return windows

_ = demonstrate_25d_windowing(num_slices=32)

## 6. Özet ve Sonraki Adım
Bu notebook ile teyit ettiğimiz bulgular:
1. **Etiketleme Zorunluluğu:** 4.349 hastanın raporundan etiket türetmeden görüntü modeli eğitmek yetersiz kalacaktır.
2. **Düzlem Odaklılık:** Sagittal, Coronal ve Axial serilerin anatomik rolleri modele ayrı ayrı bildirilmelidir.
3. **Sıradaki Adım (Faz 2):** Raporlardan LLM/kural tabanlı etiketleri yükleyip 58 altın vaka üzerinde doğrulamak!